- `mover/05.dataset_new`: 18 ≤ age

- `mover/05.dataset_18to65`: 18 ≤ age < 65

- `mover/05.dataset_65`: 65 ≤ age

- `mover/05.dataset_male`: only male

- `mover/05.dataset_female`: only female

In [1]:
import joblib
import logging
from pathlib import Path
import numpy as np
import numpy.typing as npt
from tqdm import tqdm
from xgboost import XGBClassifier
from keras.models import load_model
from sklearn.metrics import roc_auc_score
from src.module.utils import load_npy, load_config, get_peak_trough, process_meanbp

In [2]:
def load_path(src_path, strategy="hypophetversion2"):
    pos_path = np.array(
        sorted(src_path.rglob(f"{strategy}_adj{config['adjacency']}_pos_*.npy"))
    )
    neg_path = np.array(
        sorted(src_path.rglob(f"{strategy}_adj{config['adjacency']}_neg_*.npy"))
    )
    return pos_path, neg_path


def load_dataset(pos_path, neg_path) -> tuple[npt.NDArray, npt.NDArray]:
    pos_cid = load_npy(pos_path[0])
    neg_cid = load_npy(neg_path[0])
    # env['pos_path'] : pos_lt
    pos_seg = load_npy(pos_path[2])
    neg_seg = load_npy(neg_path[1])
    pos_ts = load_npy(pos_path[3])
    neg_ts = load_npy(neg_path[2])
    pos_y, neg_y = np.ones(len(pos_seg)), np.zeros(len(neg_seg))
    X, y = np.vstack([pos_seg, neg_seg]), np.hstack([pos_y, neg_y])
    cid = np.hstack([pos_cid, neg_cid])
    ts = np.vstack([pos_ts, neg_ts])
    logging.info(f"load_dataset(pos/neg): {y.size}({pos_y.size}/{neg_y.size})")
    print(X.shape, y.shape, cid.shape, ts.shape)
    return cid, ts, X, y

In [3]:
def extract_lastmbp(arr, config):
    mbp = process_meanbp(arr.reshape(-1), config)
    mbp = np.unique(mbp[np.where(mbp != 0)])
    return mbp[-1]


def get_min_size(x, p, t):
    if x[p].size < x[t].size:
        min_size = x[p].size
    else:
        min_size = x[t].size
    return min_size


def summary_stat(arr):
    x = arr.reshape(-1)
    x_mean, x_std, x_median, x_min, x_max = (
        np.mean(x),
        np.std(x),
        np.median(x),
        np.min(x),
        np.max(x),
    )
    return x_mean, x_std, x_median, x_min, x_max


def slope_intercept(t, y):
    t = np.asarray(t, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    tm, ym = t.mean(), y.mean()
    dt = t - tm
    denom = (dt * dt).sum()
    if denom == 0:
        return np.nan, np.nan
    slope = (dt * (y - ym)).sum() / denom
    return slope, ym - (slope * tm)


def extract_feature(arr, config):
    feature = []
    p, t = get_peak_trough(arr, config["detection"])
    min_size = get_min_size(arr, p, t)
    mbp = process_meanbp(arr.reshape(-1), config)
    sbp, dbp, pp = arr[p], arr[t], arr[p][:min_size] - arr[t][:min_size]
    mbp = np.unique(mbp[np.where(mbp != 0)])

    sbp_stat = summary_stat(sbp)
    dbp_stat = summary_stat(dbp)
    pp_stat = summary_stat(pp)
    mbp_stat = summary_stat(mbp)
    trend_per_beat, _ = slope_intercept(np.arange(mbp.size), mbp)
    trend_per_sec = trend_per_beat * (mbp.size / 30.0)
    half = mbp.size // 2
    slope_per_beat, _ = slope_intercept(np.arange(mbp.size)[half:], mbp[half:])
    slope_per_sec = slope_per_beat * (mbp.size / 30.0)
    last_mbp = mbp[-1]

    feature.extend(
        [
            *sbp_stat,
            *dbp_stat,
            *pp_stat,
            *mbp_stat,
            trend_per_sec,
            slope_per_sec,
            last_mbp,
        ]
    )
    return feature


def scale_input(input) -> npt.NDArray:
    return input.astype(np.float32) / 200

In [4]:
def score_sampled_auroc(idx_by_pat, true_y, prob_y):
    pat_ids = np.array(list(idx_by_pat.keys()))
    sampled = np.random.choice(pat_ids, size=len(pat_ids), replace=True)
    idx = np.concatenate([idx_by_pat[p] for p in sampled])
    sampled_y = true_y[idx]
    sampled_mbp = prob_y[idx]
    auroc = roc_auc_score(sampled_y, sampled_mbp)
    return auroc

In [5]:
config = load_config("../../config/conf.yaml")
data_path = Path(config["data_path"]).expanduser() / "mover"
model_path = Path(config["model_path"]).expanduser() / "revision_seed42"
# pos_path_new, neg_path_new = load_path(data_path / "05.dataset_new")
pos_path_18to65, neg_path_18to65 = load_path(data_path / "05.dataset_18to65")
pos_path_65, neg_path_65 = load_path(data_path / "05.dataset_65")
pos_path_female, neg_path_female = load_path(data_path / "05.dataset_female")
pos_path_male, neg_path_male = load_path(data_path / "05.dataset_male")

In [6]:
# cid_new, ts_new, X_new, y_new = load_dataset(pos_path_new, neg_path_new)
# print(
#     np.unique(cid_new).shape,
#     cid_new.shape,
#     X_new.shape,
#     y_new.shape,
#     sum(y_new == 1),
#     sum(y_new == 0),
#     np.mean(y_new),
# )

cid_18to65, ts_18to65, X_18to65, y_18to65 = load_dataset(
    pos_path_18to65, neg_path_18to65
)
print(
    np.unique(cid_18to65).shape,
    cid_18to65.shape,
    X_18to65.shape,
    y_18to65.shape,
    sum(y_18to65 == 1),
    sum(y_18to65 == 0),
    np.mean(y_18to65),
)

cid_65, ts_65, X_65, y_65 = load_dataset(pos_path_65, neg_path_65)
print(
    np.unique(cid_65).shape,
    cid_65.shape,
    X_65.shape,
    y_65.shape,
    sum(y_65 == 1),
    sum(y_65 == 0),
    np.mean(y_65),
)

cid_female, ts_female, X_female, y_female = load_dataset(
    pos_path_female, neg_path_female
)
print(
    np.unique(cid_female).shape,
    cid_female.shape,
    X_female.shape,
    y_female.shape,
    sum(y_female == 1),
    sum(y_female == 0),
    np.mean(y_female),
)

cid_male, ts_male, X_male, y_male = load_dataset(pos_path_male, neg_path_male)
print(
    np.unique(cid_male).shape,
    cid_male.shape,
    X_male.shape,
    y_male.shape,
    sum(y_male == 1),
    sum(y_male == 0),
    np.mean(y_male),
)

(13197, 3000) (13197,) (13197,) (13197, 3000)
(348,) (13197,) (13197, 3000) (13197,) 3843 9354 0.29120254603318935
(12785, 3000) (12785,) (12785,) (12785, 3000)
(307,) (12785,) (12785, 3000) (12785,) 5468 7317 0.42768869769260853
(11543, 3000) (11543,) (11543,) (11543, 3000)
(310,) (11543,) (11543, 3000) (11543,) 3900 7643 0.33786710560512867
(14250, 3000) (14250,) (14250,) (14250, 3000)
(339,) (14250,) (14250, 3000) (14250,) 5395 8855 0.37859649122807015


In [7]:
# 18 ≤ age < 65; 65 ≤ age 의 수를 합쳤을 때 18 ≤ age 의 수가 안 나오는 이유:
# 동일 환자가 여러 차례 방문하면서 나이가 64에서 65로 넘어간 경우가 6명 있음.
np.intersect1d(np.unique(cid_18to65), np.unique(cid_65))

array(['06d831f60cb0da4c', '0ad5e555089d434e', '5ac036375932b8f0',
       '6e4c591bab352c98', '8572326e43096a09', 'a931e6d17a6c9801'],
      dtype='<U16')

In [8]:
# X_new_feat = np.array([extract_feature(x, config) for x in tqdm(X_new, ncols=75)])
# print(X_new_feat.shape)
X_18to65_feat = np.array([extract_feature(x, config) for x in tqdm(X_18to65, ncols=75)])
print(X_18to65_feat.shape)
X_65_feat = np.array([extract_feature(x, config) for x in tqdm(X_65, ncols=75)])
print(X_65_feat.shape)
X_female_feat = np.array([extract_feature(x, config) for x in tqdm(X_female, ncols=75)])
print(X_female_feat.shape)
X_male_feat = np.array([extract_feature(x, config) for x in tqdm(X_male, ncols=75)])
print(X_male_feat.shape)

100%|██████████████████████████████| 13197/13197 [00:05<00:00, 2554.37it/s]


(13197, 23)


100%|██████████████████████████████| 12785/12785 [00:04<00:00, 2643.16it/s]


(12785, 23)


100%|██████████████████████████████| 11543/11543 [00:04<00:00, 2781.86it/s]


(11543, 23)


100%|██████████████████████████████| 14250/14250 [00:04<00:00, 2852.47it/s]

(14250, 23)


In [9]:
X_18to65_s = scale_input(X_18to65)
X_65_s = scale_input(X_65)
X_female_s = scale_input(X_female)
X_male_s = scale_input(X_male)

# MBP Baseline

In [10]:
# eval 목적
lastmbp_18to65 = np.array(
    [extract_lastmbp(x, config) for x in tqdm(X_18to65, ncols=75)]
)
print(lastmbp_18to65.shape)
lastmbp_65 = np.array([extract_lastmbp(x, config) for x in tqdm(X_65, ncols=75)])
print(lastmbp_65.shape)
lastmbp_female = np.array(
    [extract_lastmbp(x, config) for x in tqdm(X_female, ncols=75)]
)
print(lastmbp_female.shape)
lastmbp_male = np.array([extract_lastmbp(x, config) for x in tqdm(X_male, ncols=75)])
print(lastmbp_male.shape)

100%|██████████████████████████████| 13197/13197 [00:02<00:00, 5181.28it/s]


(13197,)


100%|██████████████████████████████| 12785/12785 [00:02<00:00, 5290.13it/s]


(12785,)


100%|██████████████████████████████| 11543/11543 [00:02<00:00, 5291.25it/s]


(11543,)


100%|██████████████████████████████| 14250/14250 [00:02<00:00, 5302.83it/s]

(14250,)


In [11]:
print(f"AUROC(18 ≤ age < 65): {roc_auc_score(y_18to65, -lastmbp_18to65):.3f}")
print(f"AUROC(65 ≤ age): {roc_auc_score(y_65, -lastmbp_65):.3f}")
print(f"AUROC(male): {roc_auc_score(y_male, -lastmbp_male):.3f}")
print(f"AUROC(female): {roc_auc_score(y_female, -lastmbp_female):.3f}")

AUROC(18 ≤ age < 65): 0.735
AUROC(65 ≤ age): 0.673
AUROC(male): 0.701
AUROC(female): 0.689


In [12]:
# age
pat_ids_18to65 = np.unique(cid_18to65)
pat_ids_65 = np.unique(cid_65)
idx_by_pat_18to65 = {p: np.where(cid_18to65 == p)[0] for p in pat_ids_18to65}
idx_by_pat_65 = {p: np.where(cid_65 == p)[0] for p in pat_ids_65}
np.random.seed(config["seed"])  # 42

auroc_18to65_list, auroc_65_list, delta_list = [], [], []
n = 2000
for i in tqdm(range(n), ncols=75):
    auroc_18to65 = score_sampled_auroc(idx_by_pat_18to65, y_18to65, -lastmbp_18to65)
    auroc_65 = score_sampled_auroc(idx_by_pat_65, y_65, -lastmbp_65)
    auroc_18to65_list.append(auroc_18to65)
    auroc_65_list.append(auroc_65)
    delta_list.append(auroc_18to65 - auroc_65)

print("95% CI (without gray)")
print(
    f"AUROC(18 ≤ age < 65): ({np.percentile(auroc_18to65_list, 2.5):.3f}-{np.percentile(auroc_18to65_list, 97.5):.3f})"
)
print(
    f"AUROC(65 ≤ age): ({np.percentile(auroc_65_list, 2.5):.3f}-{np.percentile(auroc_65_list, 97.5):.3f})"
)
print(
    f"∆AUROC (vs. Ref): ({np.percentile(delta_list, 2.5):.3f}-{np.percentile(delta_list, 97.5):.3f})"
)

100%|█████████████████████████████████| 2000/2000 [00:06<00:00, 313.09it/s]

95% CI (without gray)
AUROC(18 ≤ age < 65): (0.704-0.766)
AUROC(65 ≤ age): (0.641-0.704)
∆AUROC (vs. Ref): (0.019-0.107)


In [13]:
# sex
pat_ids_male = np.unique(cid_male)
pat_ids_female = np.unique(cid_female)
idx_by_pat_male = {p: np.where(cid_male == p)[0] for p in pat_ids_male}
idx_by_pat_female = {p: np.where(cid_female == p)[0] for p in pat_ids_female}
np.random.seed(config["seed"])  # 42

auroc_male_list, auroc_female_list, delta_list = [], [], []
n = 2000
for i in tqdm(range(n), ncols=75):
    auroc_male = score_sampled_auroc(idx_by_pat_male, y_male, -lastmbp_male)
    auroc_female = score_sampled_auroc(idx_by_pat_female, y_female, -lastmbp_female)
    auroc_male_list.append(auroc_male)
    auroc_female_list.append(auroc_female)
    delta_list.append(auroc_male - auroc_female)

print("95% CI (without gray)")
print(
    f"AUROC(Male): ({np.percentile(auroc_male_list, 2.5):.3f}-{np.percentile(auroc_male_list, 97.5):.3f})"
)
print(
    f"AUROC(Female): ({np.percentile(auroc_female_list, 2.5):.3f}-{np.percentile(auroc_female_list, 97.5):.3f})"
)
print(
    f"∆AUROC (vs. Ref): ({np.percentile(delta_list, 2.5):.3f}-{np.percentile(delta_list, 97.5):.3f})"
)

100%|█████████████████████████████████| 2000/2000 [00:06<00:00, 315.92it/s]

95% CI (without gray)
AUROC(Male): (0.668-0.731)
AUROC(Female): (0.656-0.722)
∆AUROC (vs. Ref): (-0.034-0.058)


# LR

In [10]:
lrpipe = joblib.load(model_path / "lrpipe_baseline.joblib")

In [11]:
prob_y_18to65 = lrpipe["model"].predict_proba(X_18to65_feat)[:, 1]
prob_y_65 = lrpipe["model"].predict_proba(X_65_feat)[:, 1]
prob_y_male = lrpipe["model"].predict_proba(X_male_feat)[:, 1]
prob_y_female = lrpipe["model"].predict_proba(X_female_feat)[:, 1]

In [12]:
print(f"AUROC(18 ≤ age < 65): {roc_auc_score(y_18to65, prob_y_18to65):.3f}")
print(f"AUROC(65 ≤ age): {roc_auc_score(y_65, prob_y_65):.3f}")
print(f"AUROC(male): {roc_auc_score(y_male, prob_y_male):.3f}")
print(f"AUROC(female): {roc_auc_score(y_female, prob_y_female):.3f}")

AUROC(18 ≤ age < 65): 0.777
AUROC(65 ≤ age): 0.693
AUROC(male): 0.728
AUROC(female): 0.754


In [13]:
# age
pat_ids_18to65 = np.unique(cid_18to65)
pat_ids_65 = np.unique(cid_65)
idx_by_pat_18to65 = {p: np.where(cid_18to65 == p)[0] for p in pat_ids_18to65}
idx_by_pat_65 = {p: np.where(cid_65 == p)[0] for p in pat_ids_65}
np.random.seed(config["seed"])  # 42

auroc_18to65_list, auroc_65_list, delta_list = [], [], []
n = 2000
for i in tqdm(range(n), ncols=75):
    auroc_18to65 = score_sampled_auroc(idx_by_pat_18to65, y_18to65, prob_y_18to65)
    auroc_65 = score_sampled_auroc(idx_by_pat_65, y_65, prob_y_65)
    auroc_18to65_list.append(auroc_18to65)
    auroc_65_list.append(auroc_65)
    delta_list.append(auroc_18to65 - auroc_65)

print("95% CI (without gray)")
print(
    f"AUROC(18 ≤ age < 65): ({np.percentile(auroc_18to65_list, 2.5):.3f}-{np.percentile(auroc_18to65_list, 97.5):.3f})"
)
print(
    f"AUROC(65 ≤ age): ({np.percentile(auroc_65_list, 2.5):.3f}-{np.percentile(auroc_65_list, 97.5):.3f})"
)
print(
    f"∆AUROC (vs. Ref): ({np.percentile(delta_list, 2.5):.3f}-{np.percentile(delta_list, 97.5):.3f})"
)

  0%|                                             | 0/2000 [00:00<?, ?it/s]

100%|█████████████████████████████████| 2000/2000 [00:06<00:00, 296.64it/s]

95% CI (without gray)
AUROC(18 ≤ age < 65): (0.749-0.804)
AUROC(65 ≤ age): (0.663-0.723)
∆AUROC (vs. Ref): (0.042-0.125)


In [14]:
# sex
pat_ids_male = np.unique(cid_male)
pat_ids_female = np.unique(cid_female)
idx_by_pat_male = {p: np.where(cid_male == p)[0] for p in pat_ids_male}
idx_by_pat_female = {p: np.where(cid_female == p)[0] for p in pat_ids_female}
np.random.seed(config["seed"])  # 42

auroc_male_list, auroc_female_list, delta_list = [], [], []
n = 2000
for i in tqdm(range(n), ncols=75):
    auroc_male = score_sampled_auroc(idx_by_pat_male, y_male, prob_y_male)
    auroc_female = score_sampled_auroc(idx_by_pat_female, y_female, prob_y_female)
    auroc_male_list.append(auroc_male)
    auroc_female_list.append(auroc_female)
    delta_list.append(auroc_male - auroc_female)

print("95% CI (without gray)")
print(
    f"AUROC(Male): ({np.percentile(auroc_male_list, 2.5):.3f}-{np.percentile(auroc_male_list, 97.5):.3f})"
)
print(
    f"AUROC(Female): ({np.percentile(auroc_female_list, 2.5):.3f}-{np.percentile(auroc_female_list, 97.5):.3f})"
)
print(
    f"∆AUROC (vs. Ref): ({np.percentile(delta_list, 2.5):.3f}-{np.percentile(delta_list, 97.5):.3f})"
)

100%|█████████████████████████████████| 2000/2000 [00:06<00:00, 290.42it/s]

95% CI (without gray)
AUROC(Male): (0.697-0.757)
AUROC(Female): (0.726-0.781)
∆AUROC (vs. Ref): (-0.065-0.015)


# XGB

In [15]:
loaded_xgb = XGBClassifier()
loaded_xgb.load_model(model_path / "xgb_baseline.json")

In [16]:
prob_y_18to65 = loaded_xgb.predict_proba(X_18to65_feat)[:, 1]
prob_y_65 = loaded_xgb.predict_proba(X_65_feat)[:, 1]
prob_y_male = loaded_xgb.predict_proba(X_male_feat)[:, 1]
prob_y_female = loaded_xgb.predict_proba(X_female_feat)[:, 1]

In [17]:
print(f"AUROC(18 ≤ age < 65): {roc_auc_score(y_18to65, prob_y_18to65):.3f}")
print(f"AUROC(65 ≤ age): {roc_auc_score(y_65, prob_y_65):.3f}")
print(f"AUROC(male): {roc_auc_score(y_male, prob_y_male):.3f}")
print(f"AUROC(female): {roc_auc_score(y_female, prob_y_female):.3f}")

AUROC(18 ≤ age < 65): 0.777
AUROC(65 ≤ age): 0.694
AUROC(male): 0.731
AUROC(female): 0.752


In [18]:
# age
pat_ids_18to65 = np.unique(cid_18to65)
pat_ids_65 = np.unique(cid_65)
idx_by_pat_18to65 = {p: np.where(cid_18to65 == p)[0] for p in pat_ids_18to65}
idx_by_pat_65 = {p: np.where(cid_65 == p)[0] for p in pat_ids_65}
np.random.seed(config["seed"])  # 42

auroc_18to65_list, auroc_65_list, delta_list = [], [], []
n = 2000
for i in tqdm(range(n), ncols=75):
    auroc_18to65 = score_sampled_auroc(idx_by_pat_18to65, y_18to65, prob_y_18to65)
    auroc_65 = score_sampled_auroc(idx_by_pat_65, y_65, prob_y_65)
    auroc_18to65_list.append(auroc_18to65)
    auroc_65_list.append(auroc_65)
    delta_list.append(auroc_18to65 - auroc_65)

print("95% CI (without gray)")
print(
    f"AUROC(18 ≤ age < 65): ({np.percentile(auroc_18to65_list, 2.5):.3f}-{np.percentile(auroc_18to65_list, 97.5):.3f})"
)
print(
    f"AUROC(65 ≤ age): ({np.percentile(auroc_65_list, 2.5):.3f}-{np.percentile(auroc_65_list, 97.5):.3f})"
)
print(
    f"∆AUROC (vs. Ref): ({np.percentile(delta_list, 2.5):.3f}-{np.percentile(delta_list, 97.5):.3f})"
)

  0%|                                             | 0/2000 [00:00<?, ?it/s]

100%|█████████████████████████████████| 2000/2000 [00:13<00:00, 149.42it/s]

95% CI (without gray)
AUROC(18 ≤ age < 65): (0.750-0.804)
AUROC(65 ≤ age): (0.665-0.723)
∆AUROC (vs. Ref): (0.043-0.122)


In [19]:
# sex
pat_ids_male = np.unique(cid_male)
pat_ids_female = np.unique(cid_female)
idx_by_pat_male = {p: np.where(cid_male == p)[0] for p in pat_ids_male}
idx_by_pat_female = {p: np.where(cid_female == p)[0] for p in pat_ids_female}
np.random.seed(config["seed"])  # 42

auroc_male_list, auroc_female_list, delta_list = [], [], []
n = 2000
for i in tqdm(range(n), ncols=75):
    auroc_male = score_sampled_auroc(idx_by_pat_male, y_male, prob_y_male)
    auroc_female = score_sampled_auroc(idx_by_pat_female, y_female, prob_y_female)
    auroc_male_list.append(auroc_male)
    auroc_female_list.append(auroc_female)
    delta_list.append(auroc_male - auroc_female)

print("95% CI (without gray)")
print(
    f"AUROC(Male): ({np.percentile(auroc_male_list, 2.5):.3f}-{np.percentile(auroc_male_list, 97.5):.3f})"
)
print(
    f"AUROC(Female): ({np.percentile(auroc_female_list, 2.5):.3f}-{np.percentile(auroc_female_list, 97.5):.3f})"
)
print(
    f"∆AUROC (vs. Ref): ({np.percentile(delta_list, 2.5):.3f}-{np.percentile(delta_list, 97.5):.3f})"
)

  0%|                                             | 0/2000 [00:00<?, ?it/s]

100%|█████████████████████████████████| 2000/2000 [00:10<00:00, 182.31it/s]

95% CI (without gray)
AUROC(Male): (0.702-0.759)
AUROC(Female): (0.724-0.779)
∆AUROC (vs. Ref): (-0.059-0.018)


# CNN

In [10]:
loaded_model = load_model(model_path / "cnn.model.keras")

2026-10-08 08:35:33.316195: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M5 Max
2026-10-08 08:35:33.316230: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 128.00 GB
2026-10-08 08:35:33.316243: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 53.76 GB
2026-10-08 08:35:33.316266: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-10-08 08:35:33.316284: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [11]:
prob_y_18to65 = loaded_model.predict(X_18to65_s).reshape(-1)
prob_y_65 = loaded_model.predict(X_65_s).reshape(-1)
prob_y_male = loaded_model.predict(X_male_s).reshape(-1)
prob_y_female = loaded_model.predict(X_female_s).reshape(-1)

151/413 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step

2026-10-08 08:35:33.529632: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
400/400 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
446/446 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
361/361 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


In [12]:
print(f"AUROC(18 ≤ age < 65): {roc_auc_score(y_18to65, prob_y_18to65):.3f}")
print(f"AUROC(65 ≤ age): {roc_auc_score(y_65, prob_y_65):.3f}")
print(f"AUROC(male): {roc_auc_score(y_male, prob_y_male):.3f}")
print(f"AUROC(female): {roc_auc_score(y_female, prob_y_female):.3f}")

AUROC(18 ≤ age < 65): 0.767
AUROC(65 ≤ age): 0.691
AUROC(male): 0.727
AUROC(female): 0.736


In [13]:
# age
pat_ids_18to65 = np.unique(cid_18to65)
pat_ids_65 = np.unique(cid_65)
idx_by_pat_18to65 = {p: np.where(cid_18to65 == p)[0] for p in pat_ids_18to65}
idx_by_pat_65 = {p: np.where(cid_65 == p)[0] for p in pat_ids_65}
np.random.seed(config["seed"])  # 42

auroc_18to65_list, auroc_65_list, delta_list = [], [], []
n = 2000
for i in tqdm(range(n), ncols=75):
    auroc_18to65 = score_sampled_auroc(idx_by_pat_18to65, y_18to65, prob_y_18to65)
    auroc_65 = score_sampled_auroc(idx_by_pat_65, y_65, prob_y_65)
    auroc_18to65_list.append(auroc_18to65)
    auroc_65_list.append(auroc_65)
    delta_list.append(auroc_18to65 - auroc_65)

print("95% CI (without gray)")
print(
    f"AUROC(18 ≤ age < 65): ({np.percentile(auroc_18to65_list, 2.5):.3f}-{np.percentile(auroc_18to65_list, 97.5):.3f})"
)
print(
    f"AUROC(65 ≤ age): ({np.percentile(auroc_65_list, 2.5):.3f}-{np.percentile(auroc_65_list, 97.5):.3f})"
)
print(
    f"∆AUROC (vs. Ref): ({np.percentile(delta_list, 2.5):.3f}-{np.percentile(delta_list, 97.5):.3f})"
)

100%|█████████████████████████████████| 2000/2000 [00:06<00:00, 307.46it/s]

95% CI (without gray)
AUROC(18 ≤ age < 65): (0.738-0.795)
AUROC(65 ≤ age): (0.660-0.721)
∆AUROC (vs. Ref): (0.033-0.119)


In [14]:
# sex
pat_ids_male = np.unique(cid_male)
pat_ids_female = np.unique(cid_female)
idx_by_pat_male = {p: np.where(cid_male == p)[0] for p in pat_ids_male}
idx_by_pat_female = {p: np.where(cid_female == p)[0] for p in pat_ids_female}
np.random.seed(config["seed"])  # 42

auroc_male_list, auroc_female_list, delta_list = [], [], []
n = 2000
for i in tqdm(range(n), ncols=75):
    auroc_male = score_sampled_auroc(idx_by_pat_male, y_male, prob_y_male)
    auroc_female = score_sampled_auroc(idx_by_pat_female, y_female, prob_y_female)
    auroc_male_list.append(auroc_male)
    auroc_female_list.append(auroc_female)
    delta_list.append(auroc_male - auroc_female)

print("95% CI (without gray)")
print(
    f"AUROC(Male): ({np.percentile(auroc_male_list, 2.5):.3f}-{np.percentile(auroc_male_list, 97.5):.3f})"
)
print(
    f"AUROC(Female): ({np.percentile(auroc_female_list, 2.5):.3f}-{np.percentile(auroc_female_list, 97.5):.3f})"
)
print(
    f"∆AUROC (vs. Ref): ({np.percentile(delta_list, 2.5):.3f}-{np.percentile(delta_list, 97.5):.3f})"
)

100%|█████████████████████████████████| 2000/2000 [00:06<00:00, 313.95it/s]

95% CI (without gray)
AUROC(Male): (0.696-0.755)
AUROC(Female): (0.707-0.766)
∆AUROC (vs. Ref): (-0.051-0.031)


# ResNet

In [24]:
loaded_model = load_model(model_path / "resnet.model.keras")

2026-10-08 08:41:14.593761: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M5 Max
2026-10-08 08:41:14.593789: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 128.00 GB
2026-10-08 08:41:14.593799: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 53.76 GB
2026-10-08 08:41:14.593822: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-10-08 08:41:14.593831: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [25]:
prob_y_18to65 = loaded_model.predict(X_18to65_s).reshape(-1)
prob_y_65 = loaded_model.predict(X_65_s).reshape(-1)
prob_y_male = loaded_model.predict(X_male_s).reshape(-1)
prob_y_female = loaded_model.predict(X_female_s).reshape(-1)

2026-10-08 08:41:15.147822: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


413/413 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step
400/400 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step
446/446 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step
361/361 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step


In [26]:
print(f"AUROC(18 ≤ age < 65): {roc_auc_score(y_18to65, prob_y_18to65):.3f}")
print(f"AUROC(65 ≤ age): {roc_auc_score(y_65, prob_y_65):.3f}")
print(f"AUROC(male): {roc_auc_score(y_male, prob_y_male):.3f}")
print(f"AUROC(female): {roc_auc_score(y_female, prob_y_female):.3f}")

AUROC(18 ≤ age < 65): 0.766
AUROC(65 ≤ age): 0.692
AUROC(male): 0.727
AUROC(female): 0.749


In [27]:
# age
pat_ids_18to65 = np.unique(cid_18to65)
pat_ids_65 = np.unique(cid_65)
idx_by_pat_18to65 = {p: np.where(cid_18to65 == p)[0] for p in pat_ids_18to65}
idx_by_pat_65 = {p: np.where(cid_65 == p)[0] for p in pat_ids_65}
np.random.seed(config["seed"])  # 42

auroc_18to65_list, auroc_65_list, delta_list = [], [], []
n = 2000
for i in tqdm(range(n), ncols=75):
    auroc_18to65 = score_sampled_auroc(idx_by_pat_18to65, y_18to65, prob_y_18to65)
    auroc_65 = score_sampled_auroc(idx_by_pat_65, y_65, prob_y_65)
    auroc_18to65_list.append(auroc_18to65)
    auroc_65_list.append(auroc_65)
    delta_list.append(auroc_18to65 - auroc_65)

print("95% CI (without gray)")
print(
    f"AUROC(18 ≤ age < 65): ({np.percentile(auroc_18to65_list, 2.5):.3f}-{np.percentile(auroc_18to65_list, 97.5):.3f})"
)
print(
    f"AUROC(65 ≤ age): ({np.percentile(auroc_65_list, 2.5):.3f}-{np.percentile(auroc_65_list, 97.5):.3f})"
)
print(
    f"∆AUROC (vs. Ref): ({np.percentile(delta_list, 2.5):.3f}-{np.percentile(delta_list, 97.5):.3f})"
)

100%|█████████████████████████████████| 2000/2000 [00:06<00:00, 293.29it/s]

95% CI (without gray)
AUROC(18 ≤ age < 65): (0.736-0.794)
AUROC(65 ≤ age): (0.658-0.724)
∆AUROC (vs. Ref): (0.029-0.118)


In [28]:
# sex
pat_ids_male = np.unique(cid_male)
pat_ids_female = np.unique(cid_female)
idx_by_pat_male = {p: np.where(cid_male == p)[0] for p in pat_ids_male}
idx_by_pat_female = {p: np.where(cid_female == p)[0] for p in pat_ids_female}
np.random.seed(config["seed"])  # 42

auroc_male_list, auroc_female_list, delta_list = [], [], []
n = 2000
for i in tqdm(range(n), ncols=75):
    auroc_male = score_sampled_auroc(idx_by_pat_male, y_male, prob_y_male)
    auroc_female = score_sampled_auroc(idx_by_pat_female, y_female, prob_y_female)
    auroc_male_list.append(auroc_male)
    auroc_female_list.append(auroc_female)
    delta_list.append(auroc_male - auroc_female)

print("95% CI (without gray)")
print(
    f"AUROC(Male): ({np.percentile(auroc_male_list, 2.5):.3f}-{np.percentile(auroc_male_list, 97.5):.3f})"
)
print(
    f"AUROC(Female): ({np.percentile(auroc_female_list, 2.5):.3f}-{np.percentile(auroc_female_list, 97.5):.3f})"
)
print(
    f"∆AUROC (vs. Ref): ({np.percentile(delta_list, 2.5):.3f}-{np.percentile(delta_list, 97.5):.3f})"
)

100%|█████████████████████████████████| 2000/2000 [00:06<00:00, 299.71it/s]

95% CI (without gray)
AUROC(Male): (0.696-0.757)
AUROC(Female): (0.717-0.779)
∆AUROC (vs. Ref): (-0.065-0.022)


# LSTMFCN

In [29]:
loaded_model = load_model(model_path / "lstmfcn.model.keras")

In [30]:
prob_y_18to65 = loaded_model.predict(X_18to65_s).reshape(-1)
prob_y_65 = loaded_model.predict(X_65_s).reshape(-1)
prob_y_male = loaded_model.predict(X_male_s).reshape(-1)
prob_y_female = loaded_model.predict(X_female_s).reshape(-1)

413/413 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step
400/400 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step
446/446 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step
361/361 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step


In [31]:
print(f"AUROC(18 ≤ age < 65): {roc_auc_score(y_18to65, prob_y_18to65):.3f}")
print(f"AUROC(65 ≤ age): {roc_auc_score(y_65, prob_y_65):.3f}")
print(f"AUROC(male): {roc_auc_score(y_male, prob_y_male):.3f}")
print(f"AUROC(female): {roc_auc_score(y_female, prob_y_female):.3f}")

AUROC(18 ≤ age < 65): 0.760
AUROC(65 ≤ age): 0.683
AUROC(male): 0.715
AUROC(female): 0.729


In [32]:
# age
pat_ids_18to65 = np.unique(cid_18to65)
pat_ids_65 = np.unique(cid_65)
idx_by_pat_18to65 = {p: np.where(cid_18to65 == p)[0] for p in pat_ids_18to65}
idx_by_pat_65 = {p: np.where(cid_65 == p)[0] for p in pat_ids_65}
np.random.seed(config["seed"])  # 42

auroc_18to65_list, auroc_65_list, delta_list = [], [], []
n = 2000
for i in tqdm(range(n), ncols=75):
    auroc_18to65 = score_sampled_auroc(idx_by_pat_18to65, y_18to65, prob_y_18to65)
    auroc_65 = score_sampled_auroc(idx_by_pat_65, y_65, prob_y_65)
    auroc_18to65_list.append(auroc_18to65)
    auroc_65_list.append(auroc_65)
    delta_list.append(auroc_18to65 - auroc_65)

print("95% CI (without gray)")
print(
    f"AUROC(18 ≤ age < 65): ({np.percentile(auroc_18to65_list, 2.5):.3f}-{np.percentile(auroc_18to65_list, 97.5):.3f})"
)
print(
    f"AUROC(65 ≤ age): ({np.percentile(auroc_65_list, 2.5):.3f}-{np.percentile(auroc_65_list, 97.5):.3f})"
)
print(
    f"∆AUROC (vs. Ref): ({np.percentile(delta_list, 2.5):.3f}-{np.percentile(delta_list, 97.5):.3f})"
)

100%|█████████████████████████████████| 2000/2000 [00:06<00:00, 291.77it/s]

95% CI (without gray)
AUROC(18 ≤ age < 65): (0.728-0.791)
AUROC(65 ≤ age): (0.651-0.717)
∆AUROC (vs. Ref): (0.029-0.122)


In [33]:
# sex
pat_ids_male = np.unique(cid_male)
pat_ids_female = np.unique(cid_female)
idx_by_pat_male = {p: np.where(cid_male == p)[0] for p in pat_ids_male}
idx_by_pat_female = {p: np.where(cid_female == p)[0] for p in pat_ids_female}
np.random.seed(config["seed"])  # 42

auroc_male_list, auroc_female_list, delta_list = [], [], []
n = 2000
for i in tqdm(range(n), ncols=75):
    auroc_male = score_sampled_auroc(idx_by_pat_male, y_male, prob_y_male)
    auroc_female = score_sampled_auroc(idx_by_pat_female, y_female, prob_y_female)
    auroc_male_list.append(auroc_male)
    auroc_female_list.append(auroc_female)
    delta_list.append(auroc_male - auroc_female)

print("95% CI (without gray)")
print(
    f"AUROC(Male): ({np.percentile(auroc_male_list, 2.5):.3f}-{np.percentile(auroc_male_list, 97.5):.3f})"
)
print(
    f"AUROC(Female): ({np.percentile(auroc_female_list, 2.5):.3f}-{np.percentile(auroc_female_list, 97.5):.3f})"
)
print(
    f"∆AUROC (vs. Ref): ({np.percentile(delta_list, 2.5):.3f}-{np.percentile(delta_list, 97.5):.3f})"
)

100%|█████████████████████████████████| 2000/2000 [00:06<00:00, 302.74it/s]

95% CI (without gray)
AUROC(Male): (0.682-0.744)
AUROC(Female): (0.692-0.762)
∆AUROC (vs. Ref): (-0.060-0.031)


# InceptionTime

In [34]:
loaded_model = load_model(model_path / "inceptiontime.model.keras")

In [35]:
prob_y_18to65 = loaded_model.predict(X_18to65_s).reshape(-1)
prob_y_65 = loaded_model.predict(X_65_s).reshape(-1)
prob_y_male = loaded_model.predict(X_male_s).reshape(-1)
prob_y_female = loaded_model.predict(X_female_s).reshape(-1)

413/413 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step
400/400 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step
446/446 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step
361/361 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step


In [36]:
print(f"AUROC(18 ≤ age < 65): {roc_auc_score(y_18to65, prob_y_18to65):.3f}")
print(f"AUROC(65 ≤ age): {roc_auc_score(y_65, prob_y_65):.3f}")
print(f"AUROC(male): {roc_auc_score(y_male, prob_y_male):.3f}")
print(f"AUROC(female): {roc_auc_score(y_female, prob_y_female):.3f}")

AUROC(18 ≤ age < 65): 0.755
AUROC(65 ≤ age): 0.674
AUROC(male): 0.719
AUROC(female): 0.734


In [37]:
# age
pat_ids_18to65 = np.unique(cid_18to65)
pat_ids_65 = np.unique(cid_65)
idx_by_pat_18to65 = {p: np.where(cid_18to65 == p)[0] for p in pat_ids_18to65}
idx_by_pat_65 = {p: np.where(cid_65 == p)[0] for p in pat_ids_65}
np.random.seed(config["seed"])  # 42

auroc_18to65_list, auroc_65_list, delta_list = [], [], []
n = 2000
for i in tqdm(range(n), ncols=75):
    auroc_18to65 = score_sampled_auroc(idx_by_pat_18to65, y_18to65, prob_y_18to65)
    auroc_65 = score_sampled_auroc(idx_by_pat_65, y_65, prob_y_65)
    auroc_18to65_list.append(auroc_18to65)
    auroc_65_list.append(auroc_65)
    delta_list.append(auroc_18to65 - auroc_65)

print("95% CI (without gray)")
print(
    f"AUROC(18 ≤ age < 65): ({np.percentile(auroc_18to65_list, 2.5):.3f}-{np.percentile(auroc_18to65_list, 97.5):.3f})"
)
print(
    f"AUROC(65 ≤ age): ({np.percentile(auroc_65_list, 2.5):.3f}-{np.percentile(auroc_65_list, 97.5):.3f})"
)
print(
    f"∆AUROC (vs. Ref): ({np.percentile(delta_list, 2.5):.3f}-{np.percentile(delta_list, 97.5):.3f})"
)

100%|█████████████████████████████████| 2000/2000 [00:06<00:00, 294.39it/s]

95% CI (without gray)
AUROC(18 ≤ age < 65): (0.723-0.787)
AUROC(65 ≤ age): (0.638-0.712)
∆AUROC (vs. Ref): (0.031-0.129)


In [38]:
# sex
pat_ids_male = np.unique(cid_male)
pat_ids_female = np.unique(cid_female)
idx_by_pat_male = {p: np.where(cid_male == p)[0] for p in pat_ids_male}
idx_by_pat_female = {p: np.where(cid_female == p)[0] for p in pat_ids_female}
np.random.seed(config["seed"])  # 42

auroc_male_list, auroc_female_list, delta_list = [], [], []
n = 2000
for i in tqdm(range(n), ncols=75):
    auroc_male = score_sampled_auroc(idx_by_pat_male, y_male, prob_y_male)
    auroc_female = score_sampled_auroc(idx_by_pat_female, y_female, prob_y_female)
    auroc_male_list.append(auroc_male)
    auroc_female_list.append(auroc_female)
    delta_list.append(auroc_male - auroc_female)

print("95% CI (without gray)")
print(
    f"AUROC(Male): ({np.percentile(auroc_male_list, 2.5):.3f}-{np.percentile(auroc_male_list, 97.5):.3f})"
)
print(
    f"AUROC(Female): ({np.percentile(auroc_female_list, 2.5):.3f}-{np.percentile(auroc_female_list, 97.5):.3f})"
)
print(
    f"∆AUROC (vs. Ref): ({np.percentile(delta_list, 2.5):.3f}-{np.percentile(delta_list, 97.5):.3f})"
)

100%|█████████████████████████████████| 2000/2000 [00:06<00:00, 300.44it/s]

95% CI (without gray)
AUROC(Male): (0.683-0.752)
AUROC(Female): (0.695-0.769)
∆AUROC (vs. Ref): (-0.065-0.035)
